# STA 160 Final Project — Warhol's Marilyn Monroe: Color Analysis

**Student:** Chiyang Chen | UC Davis STA 160, Spring 2023

A statistical color analysis of Andy Warhol’s iconic 1964 *Marilyn Monroe* silkscreen prints across five color variants (Red, Light Blue, Sega Blue, Orange, Turquoise), using image processing, K-Means clustering, conditional entropy, and HSV color manipulation.

# Environment Setup & Reading Data

## Packages

To rerun the code, please download following packages:
```python
>>> !pip install opencv-python or !pip install opencv-contrib-python
>>> !pip install rembg
```

In [1]:
# Download Packages Code Chunck

In [2]:
# ------- Basics ------- #
from PIL import Image
from numpy import asarray
import os
import warnings 
import math
import pandas as pd
from collections import Counter
import sys
import copy
import numpy as np

# ------- Data Visualization ------- #
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib import colors
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# ------- Color Related ------- #
import cv2

# ------- Models ------- #
from sklearn.cluster import KMeans
from rembg import remove

# ------- Other Settings ------- #
warnings.filterwarnings('ignore')
sns.set_style('white')
plt.rcParams['figure.dpi'] = 500
plt.rcParams["font.family"] = "serif"
pd.set_option('display.max_columns', 30)

# ------- Color ------- #
class color:
    CYAN = '\033[96m'
    BLUE = '\033[94m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    BOLD = '\033[1m'
    BOLD_CYAN_COLOR = '\033[1m' + '\033[96m'
    BOLD_RED_COLOR = '\033[1m' + '\033[91m'
    END = '\033[0m'

##  `ImageData` Class

**Structure of the ImageData class:**

```bash
                       - .rgba
                       |
PNG file -> Read image - .rgb
                       |
                       - .pillow_image
```

**To access each image's RGBA (Red, Green, Blue, Alpha where Alpha is transparency):**
```python
>>> <COLOR>_marilyn.rgba
array([[[176,  26,   6, 255],
        [178,  25,   7, 255],
        [175,  23,   6, 255],
        ...,
        [170,  36,   3, 255],
        [168,  36,   1, 255],
        [166,  34,   1, 255]]], dtype=uint8)
```

**To access each image's RGB:**
```python
>>> <COLOR>_marilyn.rgb
array([[[176,  26,   6],
        [178,  25,   7],
        [175,  23,   6],
        ...,
        [170,  36,   3],
        [168,  36,   1],
        [166,  34,   1]]], dtype=uint8)
```

**To Reshape the image to a 2D array of pixels:**
```python
>>> <COLOR>_marilyn.data.reshape(-1, 3)
array([[176,  26,   6],
       [255, 178,  25],
       [  7, 255, 175],
       ...,
       [255, 168,  36],
       [  1, 255, 166],
       [ 34,   1, 255]], dtype=uint8)
```

**To Load the image:**
```python
>>> <COLOR>_marilyn.pillow_image
```

In [3]:
# Set up ImageData class 
class ImageData():
    """ Class that contians PNG's RGBA, RGB, and Pillow Image Format
    """
    def __init__(self, png):
        self.rgba = asarray(Image.open(png))
        self.rgb = asarray(Image.open(png).convert('RGB'))
        self.pillow_image = Image.fromarray(self.rgba)

    def image_info(self):
        """Display basic info of images
        """
        print(color.BOLD  + "Summarize image details:" + color.END)
        print(f"   - RGBA: {self.rgba.shape}")
        print(f"   - RGB: {self.rgb.shape}")
        print(f"   - Default Data Type: {self.pillow_image.mode}")
        print(f"   - Image Size: {self.pillow_image.size}")
        return self.pillow_image

In [4]:
# Read image files and read it as ImageData class
images = []
for files in os.listdir(os.getcwd()):
    if '.png' in files:
        print(color.BOLD_CYAN_COLOR + " ".join([word.capitalize() for word in files[:-15].split("_")]) + ":" + color.END)
        globals()[files[5:-15]] = ImageData(files)
        images.append(files[5:-15])
        print("    - Reading image to ImageData class as" + color.BOLD_RED_COLOR + f" {files[5:-15]}"+ color.END +" completed.\n")

Shot Red Marilyn:
    - Reading image to ImageData class as red_marilyn completed.

Shot Segablue Marilyn:
    - Reading image to ImageData class as segablue_marilyn completed.

Shot Turquoise Marilyn:
    - Reading image to ImageData class as turquoise_marilyn completed.

Shot Lightblue Marilyn:
    - Reading image to ImageData class as lightblue_marilyn completed.

Shot Orange Marilyn:
    - Reading image to ImageData class as orange_marilyn completed.



---

<div class="alert alert-warning"> <b> How to access each image from the `images` list </b></div>

**List index**:
```python
>>> locals()[images[0]]
<__main__.ImageData at 0x16c0b2310>
```

**`for` loop**:
```python
>>> for i in images:
        print(type(locals()[i]))
<class '__main__.ImageData'>
<class '__main__.ImageData'>
<class '__main__.ImageData'>
<class '__main__.ImageData'>
<class '__main__.ImageData'>
```

## Preview of each image

### Red

In [5]:
red_marilyn.image_info()

Summarize image details:
   - RGBA: (750, 750, 4)
   - RGB: (750, 750, 3)
   - Default Data Type: RGBA
   - Image Size: (750, 750)


<PIL.Image.Image image mode=RGBA size=750x750>

### Light Blue

In [6]:
lightblue_marilyn.image_info()

Summarize image details:
   - RGBA: (750, 750, 3)
   - RGB: (750, 750, 3)
   - Default Data Type: RGB
   - Image Size: (750, 750)


<PIL.Image.Image image mode=RGB size=750x750>

### Sega Blue

In [7]:
segablue_marilyn.image_info()

Summarize image details:
   - RGBA: (750, 750, 4)
   - RGB: (750, 750, 3)
   - Default Data Type: RGBA
   - Image Size: (750, 750)


<PIL.Image.Image image mode=RGBA size=750x750>

### Orange

In [8]:
orange_marilyn.image_info()

Summarize image details:
   - RGBA: (750, 750, 4)
   - RGB: (750, 750, 3)
   - Default Data Type: RGBA
   - Image Size: (750, 750)


<PIL.Image.Image image mode=RGBA size=750x750>

### Turquoise

In [9]:
turquoise_marilyn.image_info()

Summarize image details:
   - RGBA: (750, 750, 4)
   - RGB: (750, 750, 3)
   - Default Data Type: RGBA
   - Image Size: (750, 750)


<PIL.Image.Image image mode=RGBA size=750x750>

### Light Blue

In [10]:
lightblue_marilyn.image_info()

Summarize image details:
   - RGBA: (750, 750, 3)
   - RGB: (750, 750, 3)
   - Default Data Type: RGB
   - Image Size: (750, 750)


<PIL.Image.Image image mode=RGB size=750x750>

## Overall

In [11]:
fig, axes = plt.subplots(1, 5, figsize=(12, 5))
for i, ax in enumerate(axes):
    # Read images based on its RGBA
    ax.imshow(locals()[images[i]].rgba)
    ax.set_xticks([])
    ax.set_yticks([])

    # Image Label setup
    ax.set_xlabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')
    
    # Adjust axis linewidth
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        
plt.subplots_adjust(wspace=0.1)
plt.show()

<Figure size 6000x2500 with 5 Axes>

# Color Analysis

## RGB & HSV Visualization - Original

In [12]:
def RGB_HSV_graph(image):
    fig = plt.figure(figsize=(12, 12))

    # RGB
    r, g, b = cv2.split(image)
    pixel_colors = image.reshape((np.shape(image)[0]*np.shape(image)[1], 3))
    norm = colors.Normalize(vmin=-1.,vmax=1.)
    norm.autoscale(pixel_colors)
    pixel_colors = norm(pixel_colors).tolist()
    axis1 = fig.add_subplot(1, 2, 1, projection="3d")
    axis1.scatter(r.flatten(), g.flatten(), b.flatten(), facecolors=pixel_colors, marker=".")
    axis1.set_title("A",fontweight='bold',y= -0.06)

    # HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    h, s, v = cv2.split(hsv)
    axis2 = fig.add_subplot(1, 2, 2, projection="3d")
    axis2.scatter(h.flatten(), s.flatten(), v.flatten(), facecolors=pixel_colors, marker=".")
    
    # Colored Scatter plot in HSV
    axis2.set_title("B",fontweight='bold',y = -0.06)
    
    fig.tight_layout()
    fig.show()

In [13]:
 RGB_HSV_graph(locals()[images[0]].rgb)

<Figure size 6000x6000 with 2 Axes>

In [14]:
 RGB_HSV_graph(locals()[images[1]].rgb)

<Figure size 6000x6000 with 2 Axes>

In [15]:
 RGB_HSV_graph(locals()[images[2]].rgb)

<Figure size 6000x6000 with 2 Axes>

In [16]:
 RGB_HSV_graph(locals()[images[3]].rgb)

<Figure size 6000x6000 with 2 Axes>

In [17]:
 RGB_HSV_graph(locals()[images[4]].rgb)

<Figure size 6000x6000 with 2 Axes>

## Interactive 3D Plot

In [18]:
def RGB_Color_3D_Plot(image):
    # rgb split
    r, g, b = cv2.split(image)
    pixel_colors = image.reshape((np.shape(image)[0]*np.shape(image)[1], 3))
    norm = colors.Normalize(vmin=-1., vmax=1.)
    norm.autoscale(pixel_colors)
    pixel_colors = norm(pixel_colors).tolist()
    fig = go.Figure(data=go.Scatter3d(
        x=r.flatten(),
        y=g.flatten(),
        z=b.flatten(),
        mode='markers',
        marker=dict(
            size=2,
            color=pixel_colors,
            opacity=1
        ),
        name="RGB"
    ))

    # Layout
    fig.update_layout(
        scene=dict(
            xaxis_title='Red',
            yaxis_title='Green',
            zaxis_title='Blue'
        ),
        width=800,
        height=800
    )
    fig.show()

<div class="alert alert-warning"> <b> Due to limited capacity of computer, our 3D interactive plot will generate separately or generate in HTML file.</b></div>

In [115]:
#RGB_Color_3D_Plot(locals()[images[0]].rgb)

In [25]:
#RGB_Color_3D_Plot(locals()[images[1]].rgb)

In [26]:
#RGB_Color_3D_Plot(locals()[images[2]].rgb)

In [27]:
#RGB_Color_3D_Plot(locals()[images[3]].rgb)

In [28]:
#RGB_Color_3D_Plot(locals()[images[4]].rgb)

In [135]:
import cv2
import numpy as np
import plotly.graph_objects as go
from matplotlib import colors

def HSV_Color_3D_Plot(image, filename):
    # Convert image to HSV
    hsv_image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    # hsv split
    h, s, v = cv2.split(hsv_image)

    # Normalize pixel colors
    pixel_colors = hsv_image.reshape((np.shape(hsv_image)[0]*np.shape(hsv_image)[1], 3))
    norm = colors.Normalize(vmin=-1.,vmax=1.)
    norm.autoscale(pixel_colors)
    pixel_colors = norm(pixel_colors).tolist()

    # Create a 3D scatter plot
    fig = go.Figure(data=go.Scatter3d(
        x=h.flatten(),
        y=s.flatten(),
        z=v.flatten(),
        mode='markers',
        marker=dict(
            size=2,
            color=pixel_colors,  # set color to an array/list of desired values
            colorscale='Viridis',  # choose a colorscale
            opacity=1
        ),
        name="HSV"
    ))

    # Layout
    fig.update_layout(
        scene=dict(
            xaxis_title='Hue',
            yaxis_title='Saturation',
            zaxis_title='Value'
        ),
        width=800,
        height=800
    )

    # Show the plot
    fig.show()
    
    # Save the plot as HTML
    pio.write_html(fig, filename)



In [143]:
#HSV_Color_3D_Plot(locals()[images[0]].rgb, 'red_marilyn_hsv.html')

In [144]:
#HSV_Color_3D_Plot(locals()[images[1]].rgb, 'segablue_marilyn_hsv.html')

In [145]:
#HSV_Color_3D_Plot(locals()[images[2]].rgb, 'turquoise_marilyn_hsv.html')

In [146]:
#HSV_Color_3D_Plot(locals()[images[3]].rgb, 'lightblue_marilyn_hsv.html')

In [147]:
#HSV_Color_3D_Plot(locals()[images[4]].rgb, 'orange_marilyn_hsv.html')

## Conditional Entropy

### Formula

**Following method has reference:**

Relative conditional entropy values between the red, green, and blue coordinates of all pixels.

Define a digital image $I$ as the set $S(I)$ of its pixels. Let $N$ be the cardinality of that set, namely the total number of pixels in the image. Let $C$ be one of the fundamental colors, namely $C$ can be $R$, $G$, or $B$.

Let $S(c)$ be the set of pixels such that its color coordinate for the color $C$ is $c$:

$$
S(c) = \{p \in S(I) | C(p) = c \}
$$

The probability of observing color $C$ with intensity $c$ within the image is then given by:

$$
P(C = c) = \frac{|S(c)|}{N}
$$ where $|S(c)|$ stands for the number of elements of set.

The entropy of the color $C$ within the image is then given by:

$$
H(C) = - \sum^{255}_{c = 0} P(C = c) \log(P(C = c))
$$

> The entropy measures the amount of “information” associated with the color $C$ in the image. If the color $C$ is always represented with the same value over each pixel, the entropy is zero, while if the possible values for the color are evenly distributed, the entropy is at its maximum with a value of $\log(256)$.


**The association of fundamental colors within the image**

Let $C$ and $D$ be two of the three fundamental colors. Let $P(C = c, D = d)$ be the joint probability of those colors $C$ and $D$ taking the values $c$ and $d$, respectively. 

Define the set $S(c,d)$ as 

$$
S(c,d) = \{p \in S(I) | C(p) = c \,\, \text{&} \,\, D(p) = d \}
$$

then

$$
P(C = c, D = d) = \frac{|S(c,d)|}{N}
$$

The conditional probability that $C = c$, knowing that $D = d$, as
$$
P(C = c|D = d) = \frac{P(C = c, D = d)}{P(d)}
$$

Then the entropy is computed as:

$$
\begin{aligned}
H(C|D = d) &= -\sum^{255}_{c=0} P(C = c|D = d)\log (P(C = c|D = d))\\
&= -\sum^{255}_{c=0} \frac{P(C=c,D=d)}{P(D=d)} \log \frac{P(C=c,D=d)}{P(D=d)}
\end{aligned}
$$

where the sums extend over all $256$ possible values for the coordinates associated to the color $C$. 

The entropy of the color $C$ conditioned on the color $D$ is then computed as the average of those numbers over all possible values for $d$:

$$
\begin{aligned}
H(C|D) &= \sum^{255}_{d=0}P(D=d)H(C|D = d)  \\
&= -\sum^{255}_{d=0} P(D=d) \sum^{255}_{c=0} \frac{P(C=c,D=d)}{P(D=d)} \log \frac{P(C=c,D=d)}{P(D=d)}\\
&= -\sum^{255}_{d=0} \sum^{255}_{c=0} P(C=c, D=d) \log \frac{P(C = c, D=d)}{P(D = d)}\\
\end{aligned}
$$

As a result, the relative conditional entropy is 

$$
HR(C|D) = \frac{H(C|D)}{H(C)}
$$

$HR(C|D)$ takes values between $0$ and $1$. $HR(C|D) = 0$ if and only if the values of $C$ are completely determined by the values of $D$. Conversely, $HR(C|D) = 1$ if and only if the values of $C$ and $D$ are independent of each other. 

### Function

In [29]:
def calculate_entropy(image, color_c):
    assert type(color_c) == str
    
    chennel_dict = {'r':0, 'g':1, 'b':2}
    channel_c = chennel_dict[color_c]
    total_pixels = np.prod(image.shape[:2])
    
    entropy = 0
    for i in range(256):
        count_c = np.sum(image[:, :, channel_c] == i)
        p_c = count_c / total_pixels
        entropy += - p_c * math.log(p_c) if p_c > 0 else 0    
    return entropy

def calculate_conditional_entropy(image, color_c, color_c_val, color_d, color_d_val):
    assert type(color_c) == str
    assert type(color_d) == str
    
    chennel_dict = {'r':0, 'g':1, 'b':2}
    channel_c = chennel_dict[color_c]
    channel_d = chennel_dict[color_d]
    
    total_pixels = np.prod(image.shape[:2])
    count_cd = np.sum(np.logical_and(image[:, :, channel_c] == color_c_val, image[:, :, channel_d] == color_d_val))
    count_d = np.sum(image[:, :, channel_d] == color_d_val)
    p_cd = count_cd / total_pixels
    p_d = count_d / total_pixels
    
    entropy_cd = -p_cd * math.log(p_cd/p_d) if p_cd > 0 else 0
    return entropy_cd

def CE(image, color_c, color_d):
    color_c_entropy = calculate_entropy(image, color_c)
    ce = 0
    for i in range(256):
        for j in range(256):
            ce += calculate_conditional_entropy(image, color_c,j, color_d,i)
    ce = ce / color_c_entropy
    return ce

In [30]:
def Conditional_Entropy(image):
    # Conditional Entropy Matrix
    ce_matrix = np.zeros((3, 3))
    ce_matrix[0,1] = CE(image, 'r', 'g')
    ce_matrix[1,0] = ce_matrix[0,1]
    ce_matrix[0,2] = CE(image, 'r', 'b')
    ce_matrix[2,0] = ce_matrix[0,2]
    ce_matrix[1,2] = CE(image, 'g', 'b')
    ce_matrix[2,1] = ce_matrix[1,2]
    
    # Plot
    classes = ['Red', 'Green', 'Blue']
    fig, ax = plt.subplots()
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)

    # Create the heatmap
    heatmap = ax.imshow(ce_matrix, cmap='Reds')

    # Add text annotations to the heatmap
    for i in range(ce_matrix.shape[0]):
        for j in range(ce_matrix.shape[1]):
            text = ax.text(j, i, f'{ce_matrix[i, j]:.2f}', ha="center", va="center", color="black")

    return plt

In [31]:
Conditional_Entropy(locals()[images[0]].rgb).show()

<Figure size 3200x2400 with 1 Axes>

In [32]:
Conditional_Entropy(locals()[images[1]].rgb).show()

<Figure size 3200x2400 with 1 Axes>

In [33]:
Conditional_Entropy(locals()[images[2]].rgb).show()

<Figure size 3200x2400 with 1 Axes>

In [34]:
Conditional_Entropy(locals()[images[3]].rgb).show()

<Figure size 3200x2400 with 1 Axes>

In [35]:
Conditional_Entropy(locals()[images[4]].rgb).show()

<Figure size 3200x2400 with 1 Axes>

## RGB Density Curves

> The purpose of the RGB density curve is to provide insights into the color distribution within an image. By analyzing the curve, you can gain a better understanding of the color composition and characteristics of the image

In [36]:
def rgb_density_curve(image_array):
    red_channel = image_array[:, :, 0]
    green_channel = image_array[:, :, 1]
    blue_channel = image_array[:, :, 2]
    red_hist, _ = np.histogram(red_channel.flatten(), bins=256, range=[0, 256])
    green_hist, _ = np.histogram(green_channel.flatten(), bins=256, range=[0, 256])
    blue_hist, _ = np.histogram(blue_channel.flatten(), bins=256, range=[0, 256])
    red_hist = red_hist / np.sum(red_hist)
    green_hist = green_hist / np.sum(green_hist)
    blue_hist = blue_hist / np.sum(blue_hist)

    plt.figure(figsize=(10, 5))
    plt.plot(red_hist, color='red', label='Red')
    plt.plot(green_hist, color='green', label='Green')
    plt.plot(blue_hist, color='blue', label='Blue')
    plt.xlabel('Pixel Intensity')
    plt.ylabel('Density')
    plt.title('RGB Density Curves')
    plt.legend(frameon=False)
    plt.show()

In [37]:
for i in images:
    rgb_density_curve(locals()[i].rgba)

<Figure size 5000x2500 with 1 Axes>

<Figure size 5000x2500 with 1 Axes>

<Figure size 5000x2500 with 1 Axes>

<Figure size 5000x2500 with 1 Axes>

<Figure size 5000x2500 with 1 Axes>

# Identification of Regions of Interest

> A ROI is defined according to its position in the image as well as from its color consistency.

In [38]:
def RGB_graph(image):
    fig = plt.figure(figsize=(12, 12))
    r, g, b = cv2.split(image)
    pixel_colors = image.reshape((np.shape(image)[0]*np.shape(image)[1], 3))
    norm = colors.Normalize(vmin=-1., vmax=1.)
    norm.autoscale(pixel_colors)
    pixel_colors = norm(pixel_colors).tolist()
    axis = fig.add_subplot(1, 2, 1, projection="3d")
    axis.scatter(r.flatten(), g.flatten(), b.flatten(), facecolors=pixel_colors, marker=".")
    plt.show()


In [39]:
def GrayScale_RGB_3plot(result):
    # Create a figure with subplots
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Plot red vs green (blue in grayscale)
    red_channel = result[:, :, 2]
    green_channel = result[:, :, 1]
    axes[0].scatter(red_channel.flatten(), green_channel.flatten(), c=red_channel.flatten(), cmap='gray')
    axes[0].set_xlabel('Red')
    axes[0].set_ylabel('Green')
    axes[0].set_title('Red vs Green')

    # Plot red vs blue (green in grayscale)
    blue_channel = result[:, :, 0]
    axes[1].scatter(red_channel.flatten(), blue_channel.flatten(), c=green_channel.flatten(), cmap='gray')
    axes[1].set_xlabel('Red')
    axes[1].set_ylabel('Blue')
    axes[1].set_title('Red vs Blue')

    # Plot green vs blue (red in grayscale)
    axes[2].scatter(green_channel.flatten(), blue_channel.flatten(), c=red_channel.flatten(), cmap='gray')
    axes[2].set_xlabel('Green')
    axes[2].set_ylabel('Blue')
    axes[2].set_title('Green vs Blue')

    fig.tight_layout()
    plt.show()

In [40]:
def Color_Extraction(image, low_range, high_range):
    low_range = np.array(low_range)
    high_range = np.array(high_range)
    mask = cv2.inRange(image, low_range, high_range)
    result = cv2.bitwise_and(image, image, mask=mask)
    return result

In [41]:
def Calculate_Proportion(image, low_range, high_range):
    low_range = np.array(low_range)
    high_range = np.array(high_range)
    mask = cv2.inRange(image, low_range, high_range)
    total_pixels = np.prod(image.shape[:2])
    matched_pixels = np.count_nonzero(mask)
    color_proportion = matched_pixels / total_pixels
    return color_proportion

## Background Proportions

### Color Threshold Method

In [42]:
background_dict = {}
fig, axes = plt.subplots(1, 5, figsize=(12, 5))
for i, ax in enumerate(axes):
    image = locals()[images[i]].rgb
    modified_image = image.copy()

    size = 360  # Size of the square (width and height)
    black_square = np.zeros((size, size, 3), dtype=np.uint8)
    black_square[:] = (0, 0, 0)
    x = 180 
    y = 280
    black_square_resized = cv2.resize(black_square, (size, size))
    modified_image[y:y+size, x:x+size] = black_square_resized
    
    background_dict[images[i]] = modified_image
    
    ax.imshow(modified_image)
    ax.set_xticks([])
    ax.set_yticks([])

    # Image Label setup
    ax.set_xlabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')
    
    # Adjust axis linewidth
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        
plt.subplots_adjust(wspace=0.1)
plt.show()

<Figure size 6000x2500 with 5 Axes>

In [43]:
background = {}
background[images[0]] = Color_Extraction(background_dict[images[0]], [140, 0, 0], [220, 60, 80])
background[images[1]] = Color_Extraction(background_dict[images[1]], [70, 188, 100], [180, 230, 230])
background[images[2]] = Color_Extraction(background_dict[images[2]], [0, 155, 10], [150, 255, 230])
background[images[3]] = Color_Extraction(background_dict[images[3]], [0, 135, 0], [90, 255, 240])
background[images[4]] = Color_Extraction(background_dict[images[4]], [200, 0, 0], [1200, 130, 100])

In [44]:
fig, axes = plt.subplots(1, 5, figsize=(12, 5))
for i, ax in enumerate(axes):
    ax.imshow(background[images[i]])
    ax.set_xticks([])
    ax.set_yticks([])

    # Image Label setup
    ax.set_xlabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')
    
    # Adjust axis linewidth
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        
plt.subplots_adjust(wspace=0.1)
plt.show()

<Figure size 6000x2500 with 5 Axes>

### Background Color Proportions

In [45]:
bg_color_prop = {}
bg_color_prop[images[0]] = Calculate_Proportion(background_dict[images[0]], [140, 0, 0], [220, 60, 80])
bg_color_prop[images[1]] = Calculate_Proportion(background_dict[images[1]], [70, 188, 100], [180, 230, 230])
bg_color_prop[images[2]] = Calculate_Proportion(background_dict[images[2]], [0, 155, 10], [150, 255, 230])
bg_color_prop[images[3]] = Calculate_Proportion(background_dict[images[3]], [0, 135, 0], [90, 255, 240])
bg_color_prop[images[4]] = Calculate_Proportion(background_dict[images[4]], [200, 0, 0], [1200, 130, 100])

In [46]:
bg_color_prop

{'red_marilyn': 0.2822648888888889,
 'segablue_marilyn': 0.32178311111111113,
 'turquoise_marilyn': 0.32064533333333334,
 'lightblue_marilyn': 0.3380568888888889,
 'orange_marilyn': 0.35612266666666664}

### Comparison

In [47]:
GrayScale_RGB_3plot(background[images[0]])

<Figure size 7500x2500 with 3 Axes>

In [48]:
GrayScale_RGB_3plot(background[images[1]])

<Figure size 7500x2500 with 3 Axes>

In [49]:
GrayScale_RGB_3plot(background[images[2]])

<Figure size 7500x2500 with 3 Axes>

In [50]:
GrayScale_RGB_3plot(background[images[3]])

<Figure size 7500x2500 with 3 Axes>

In [51]:
GrayScale_RGB_3plot(background[images[4]])

<Figure size 7500x2500 with 3 Axes>

### Deep Learning Method

In [52]:
Bg_remove = {}

In [53]:
fig, axes = plt.subplots(1, 5, figsize=(12, 5))
for i, ax in enumerate(axes):
    # Read images based on its RGBA
    img = remove(locals()[images[i]].rgb)
    ax.imshow(img)
    ax.set_xticks([])
    ax.set_yticks([])

    # Image Label setup
    ax.set_xlabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')
    
    # Add to dict
    Bg_remove[images[i]] = img
    
    # Adjust axis linewidth
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        
plt.subplots_adjust(wspace=0.1)
plt.show()

100%|████████████████████████████████████████| 176M/176M [00:00<00:00, 500GB/s]


<Figure size 6000x2500 with 5 Axes>

In [54]:
"""
https://stackoverflow.com/questions/50331463/convert-rgba-to-rgb-in-python
"""

def rgba2rgb( rgba, background=(255,255,255) ):
    row, col, ch = rgba.shape

    if ch == 3:
        return rgba

    assert ch == 4, 'RGBA image has 4 channels.'

    rgb = np.zeros( (row, col, 3), dtype='float32' )
    r, g, b, a = rgba[:,:,0], rgba[:,:,1], rgba[:,:,2], rgba[:,:,3]

    a = np.asarray( a, dtype='float32' ) / 255.0

    R, G, B = background

    rgb[:,:,0] = r * a + (1.0 - a) * R
    rgb[:,:,1] = g * a + (1.0 - a) * G
    rgb[:,:,2] = b * a + (1.0 - a) * B

    return np.asarray( rgb, dtype='uint8' )

In [55]:
for img in images:
    image = rgba2rgb(Bg_remove[img])
    white_pixels = np.sum(image == [255, 255, 255])
    total_pixels = np.prod(image.shape[:3])
    white_area_proportion = white_pixels / total_pixels
    print("Proportion of background for" + color.BOLD_RED_COLOR + f" {img}"+ color.END + ": {:.2%}".format(white_area_proportion))

Proportion of background for red_marilyn: 26.23%
Proportion of background for segablue_marilyn: 31.37%
Proportion of background for turquoise_marilyn: 29.60%
Proportion of background for lightblue_marilyn: 33.90%
Proportion of background for orange_marilyn: 33.57%


## Hair Color Analysis

In [56]:
hair_color = {}
hair_color[images[0]] = Color_Extraction(locals()[images[0]].rgb, [80, 130, 0], [255, 255, 60])
hair_color[images[1]] = Color_Extraction(locals()[images[1]].rgb, [50, 110, 0], [255, 255, 90])
hair_color[images[2]] = Color_Extraction(locals()[images[2]].rgb, [0, 165, 0], [255, 255, 140])
hair_color[images[3]] = Color_Extraction(locals()[images[3]].rgb, [0, 170, 0], [255, 255, 155])
hair_color[images[4]] = Color_Extraction(locals()[images[4]].rgb, [180, 150, 0], [255, 255, 150])

In [57]:
fig, axes = plt.subplots(1, 5, figsize=(12, 5))
for i, ax in enumerate(axes):
    ax.imshow(hair_color[images[i]])
    ax.set_xticks([])
    ax.set_yticks([])

    # Image Label setup
    ax.set_xlabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')
    
    # Adjust axis linewidth
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        
plt.subplots_adjust(wspace=0.1)
plt.show()

<Figure size 6000x2500 with 5 Axes>

### Hair Color Proportions

In [58]:
hair_color_prop = {}
hair_color_prop[images[0]] = Calculate_Proportion(locals()[images[0]].rgb, [80, 130, 0], [255, 255, 60])
hair_color_prop[images[1]] = Calculate_Proportion(locals()[images[1]].rgb, [50, 110, 0], [255, 255, 90])
hair_color_prop[images[2]] = Calculate_Proportion(locals()[images[2]].rgb, [0, 165, 0], [255, 255, 140])
hair_color_prop[images[3]] = Calculate_Proportion(locals()[images[3]].rgb, [0, 170, 0], [255, 255, 155])
hair_color_prop[images[4]] = Calculate_Proportion(locals()[images[4]].rgb, [180, 150, 0], [255, 255, 150])

In [59]:
hair_color_prop

{'red_marilyn': 0.2203342222222222,
 'segablue_marilyn': 0.21229155555555557,
 'turquoise_marilyn': 0.22644088888888889,
 'lightblue_marilyn': 0.16676622222222223,
 'orange_marilyn': 0.19496}

### Comparison

In [60]:
GrayScale_RGB_3plot(hair_color[images[0]])

<Figure size 7500x2500 with 3 Axes>

In [61]:
GrayScale_RGB_3plot(hair_color[images[1]])

<Figure size 7500x2500 with 3 Axes>

In [62]:
GrayScale_RGB_3plot(hair_color[images[2]])

<Figure size 7500x2500 with 3 Axes>

In [63]:
GrayScale_RGB_3plot(hair_color[images[3]])

<Figure size 7500x2500 with 3 Axes>

In [64]:
GrayScale_RGB_3plot(hair_color[images[4]])

<Figure size 7500x2500 with 3 Axes>

## Eyeshadow Color Analysis

In [65]:
eyeshadow_area = {}
fig, axes = plt.subplots(1, 5, figsize=(12, 5))

for i, ax in enumerate(axes):
    img = locals()[images[i]].rgb
    
    # Selected area
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    x, y, w, h = 130, 325, 400, 78 
    mask[y:y+h, x:x+w] = 255 
    selected_area = cv2.bitwise_and(img, img, mask=mask)
    eyeshadow_area[images[i]] = selected_area
    ax.imshow(selected_area)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Image Label setup
    ax.set_xlabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')
        
plt.subplots_adjust(wspace=0.1)
plt.show()

<Figure size 6000x2500 with 5 Axes>

In [66]:
eyeshadow = {}
eyeshadow[images[0]] = Color_Extraction(eyeshadow_area[images[0]], [70, 155, 160], [180, 240, 255])
eyeshadow[images[1]] = Color_Extraction(eyeshadow_area[images[1]], [50, 110, 160], [200, 240, 255])
eyeshadow[images[2]] = Color_Extraction(eyeshadow_area[images[2]], [120, 100, 195], [230, 255, 255])
eyeshadow[images[3]] = Color_Extraction(eyeshadow_area[images[3]], [75, 120, 120], [145, 180, 190])
eyeshadow[images[4]] = Color_Extraction(eyeshadow_area[images[4]], [10, 80, 130], [140, 225, 255])

In [67]:
fig, axes = plt.subplots(1, 5, figsize=(12, 5))
for i, ax in enumerate(axes):
    ax.imshow(eyeshadow[images[i]])
    ax.set_xticks([])
    ax.set_yticks([])

    # Image Label setup
    ax.set_xlabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')
    
    # Adjust axis linewidth
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        
plt.subplots_adjust(wspace=0.1)
plt.show()

<Figure size 6000x2500 with 5 Axes>

### Eyeshadow Color Proportions

In [68]:
eyeshadow_prop = {}
eyeshadow_prop[images[0]] = Calculate_Proportion(eyeshadow_area[images[0]], [80, 130, 0], [255, 255, 60])
eyeshadow_prop[images[1]] = Calculate_Proportion(eyeshadow_area[images[1]], [50, 110, 0], [255, 255, 90])
eyeshadow_prop[images[2]] = Calculate_Proportion(eyeshadow_area[images[2]], [0, 165, 0], [255, 255, 140])
eyeshadow_prop[images[3]] = Calculate_Proportion(eyeshadow_area[images[3]], [0, 170, 0], [255, 255, 155])
eyeshadow_prop[images[4]] = Calculate_Proportion(eyeshadow_area[images[4]], [180, 150, 0], [255, 255, 150])

In [69]:
eyeshadow_prop

{'red_marilyn': 0.001208888888888889,
 'segablue_marilyn': 0.0007022222222222222,
 'turquoise_marilyn': 0.0016142222222222222,
 'lightblue_marilyn': 0.000592,
 'orange_marilyn': 0.0020515555555555556}

### Comparison

In [70]:
GrayScale_RGB_3plot(eyeshadow[images[0]])

<Figure size 7500x2500 with 3 Axes>

In [71]:
GrayScale_RGB_3plot(eyeshadow[images[1]])

<Figure size 7500x2500 with 3 Axes>

In [72]:
GrayScale_RGB_3plot(eyeshadow[images[2]])

<Figure size 7500x2500 with 3 Axes>

In [73]:
GrayScale_RGB_3plot(eyeshadow[images[3]])

<Figure size 7500x2500 with 3 Axes>

In [74]:
GrayScale_RGB_3plot(eyeshadow[images[4]])

<Figure size 7500x2500 with 3 Axes>

# Extract Colors

## KMeans

**In this part, we deploy:**
1. Re-generate image based on `n_cluster`
2. Color Distributions (%) with its HSV value names

**Process - KMeans model**:
1. Choose `n_cluster`
2. Fit KMeans model by using 2D reshaped data
3. Fit the KMeans model
4. Replace colors

In [75]:
# Initilization
inertia = {}
centroid = {}
percentages = {}
KMeansImages = {}

### Generate Images with different `n_cluster` 

In [76]:
def KMeans_Image_Generator(image, cluster):
    """Fit the KMeans model based on image's RGB
    After fitting the model, the model's inertia, cluster centers, 
    and percentage of clustering colors will be stored in:
        - inertia dictionary
        - centroid dictionary
        - percentage dictionary
    The generated imaged based on KMeans model will be stored in KMeansImages
    """
    
    # Convert to 2D numpy array
    image_2D = image.rgb.reshape(image.rgb.shape[0]*image.rgb.shape[1], 3)
    model = KMeans(n_clusters=cluster, random_state=0).fit(image_2D)
    image2 = image_2D.copy()
    
    # model.inertia_
    image_name = [name for name, value in globals().items() if value is image][0]
    inertia[f"{image_name}_{cluster}"] = model.inertia_
    
    # model.cluster_centers_
    centroid[f"{image_name}_{cluster}"] = model.cluster_centers_
    
    # Percentage of colors
    percent = []
    for i in range(cluster):
        percent.append(list(model.labels_).count(i) / len(list(model.labels_)))
    percentages[f"{image_name}_{cluster}"] = percent
    
    # recolor image
    for i in np.unique(model.labels_):
        image2[model.labels_==i,:] = model.cluster_centers_[i]
    
    image2 = image2.reshape(image.rgb.shape)
    KMeansImages[f"{image_name}_{cluster}"] = image2
    
    return image2

def KMeans_Diff_clt_Image_Generator(image, clt_lst):
    """Plot 10 different numbers of cluster
    """
    assert len(clt_lst) == 10
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 6), sharex=True, sharey=True)
    fig.subplots_adjust(hspace=0.5)
    
    for i, clt in enumerate(clt_lst):
        ax = axes[i // 5, i % 5]
        img = KMeans_Image_Generator(image, clt)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'Number of Cluster = {clt}', fontweight='bold')

    plt.tight_layout()
    plt.show()

In [77]:
# Set up different numbers of cluster
clt_lst = [2, 3, 4, 6, 8, 10, 12, 16, 20, 30]

In [78]:
for i in images:
    KMeans_Diff_clt_Image_Generator(locals()[i], clt_lst)

<Figure size 7500x3000 with 10 Axes>

<Figure size 7500x3000 with 10 Axes>

<Figure size 7500x3000 with 10 Axes>

<Figure size 7500x3000 with 10 Axes>

<Figure size 7500x3000 with 10 Axes>

### With Color Proportions

In [79]:
def show_image_comparison(image, cluster):
    assert cluster in clt_lst
    # Image 1: original image
    image1 = image.rgb
    image_name = [name for name, value in globals().items() if value is image][0]
    
    # Images 2: KMeans image
    for key in KMeansImages.keys():
        image_key = image_name + f"_{str(cluster)}"
        if image_key == key :
            image2 = KMeansImages[image_key]
            image2_centroid = centroid[image_key]
            image2_percent = percentages[image_key]
    
    # Plot
    fig, ax = plt.subplots(1, 3, figsize=(11,11))
    
    # Plot 1
    ax[0].imshow(image1)
    ax[0].axis('off')
    ax[0].set_title("A",fontweight='bold',y = -0.1)
    
    # Plot 2
    ax[1].imshow(image2)
    ax[1].axis('off')
    ax[1].set_title("B",fontweight='bold',y = -0.1)
    
    # Plot 3: Pie Chart
    color_name = []
    for color in image2_centroid.astype(np.uint8).reshape(1, -1, 3)[0]:
        color_name.append('#{:02x}{:02x}{:02x}'.format(int(color[0]), int(color[1]), int(color[2])))
    
    percent_row = [round(val*100, 1) for val in image2_percent]
    labels = [f"{val}%" for val in percent_row]
    
    pie, _, autotexts = ax[2].pie(percent_row, labels=labels, colors=color_name, autopct='')
    ax[2].set_title("C",fontweight='bold',y = -0.1)
    
    # Add white borders to the pie chart wedges
    for wedge in pie:
        wedge.set_edgecolor('black')
    
    fig.tight_layout()
    plt.show()

#### `cluster = 2` Color Proportions

In [80]:
for i in images:
    show_image_comparison(locals()[i], 2)

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

#### `cluster = 3` Color Proportions

In [81]:
for i in images:
    show_image_comparison(locals()[i], 3)

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

#### `cluster = 6` Color Proportions

In [82]:
for i in images:
    show_image_comparison(locals()[i], 6)

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

#### `cluster = 10` Color Proportions

In [83]:
for i in images:
    show_image_comparison(locals()[i], 10)

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

<Figure size 5500x5500 with 3 Axes>

# Others

## Inertia Curve - Comparison
> By utilizing the Inertia curve obtained from the KMeans model, we can compare the training attributes of each color.

In [84]:
# Convert inertia dictionary to dataframe
inertia_df = pd.DataFrame(columns=['Color', 'Inertia', 'Cluster'])

for key, value in inertia.items():
    row = {'Color': key.split("_")[0].capitalize(), 'Inertia': value, 'Cluster': key.split("_")[-1]}
    inertia_df = inertia_df.append(row, ignore_index=True)

In [85]:
# preview of the inertia dataframe
inertia_df.head()

,Color,Inertia,Cluster
0,Red,4.617421e+09,2
1,Red,2.486517e+09,3
2,Red,8.989004e+08,4
3,Red,4.553446e+08,6
4,Red,3.081690e+08,8


In [86]:
# Plot inertia curve for 5 images
grouped = inertia_df.groupby('Color')

fig, ax = plt.subplots(figsize=(10, 5))
for name, group in grouped:
    ax.plot(group['Cluster'], group['Inertia'], marker='o', label=f"{name} Marilyn")

ax.set_xlabel('Number of Clusters', fontweight = 'bold')
ax.set_ylabel('Inertia', fontweight = 'bold')
ax.legend(frameon=False)
plt.show()

<Figure size 5000x2500 with 1 Axes>

# Change Colors of Eyeshadow

## `COLOR_BGR2GRAY`

In [87]:
fig, axes = plt.subplots(1, 5, figsize=(12, 5))
for i, ax in enumerate(axes):
    # Read images based on its RGBA
    image = locals()[images[i]].rgb
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY))
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Image Label setup
    ax.set_xlabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')
        
plt.subplots_adjust(wspace=0.1)
plt.show()

<Figure size 6000x2500 with 5 Axes>

## `COLOR_BGR2HSV)`

In [88]:
fig, axes = plt.subplots(1, 5, figsize=(12, 5))
for i, ax in enumerate(axes):
    image = locals()[images[i]].rgb
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2HSV))
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Image Label setup
    ax.set_xlabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')
        
plt.subplots_adjust(wspace=0.1)
plt.show()

<Figure size 6000x2500 with 5 Axes>

## Hue Channel

In [89]:
fig, axes = plt.subplots(1, 5, figsize=(12, 5))
for i, ax in enumerate(axes):
    image = locals()[images[i]].rgb
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hue = hsv[:,:,0]
    ax.imshow(hue)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Image Label setup
    ax.set_xlabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')
        
plt.subplots_adjust(wspace=0.1)
plt.show()

<Figure size 6000x2500 with 5 Axes>

## Hue Histogram

In [90]:
fig, axes = plt.subplots(5, 1, figsize=(12, 15))
for i, ax in enumerate(axes):
    image = locals()[images[i]].rgb
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hue = hsv[:,:,0]
    
    # calculate a histogram with 180 bins, one for each color
    hist, _ = np.histogram(hue, bins=180, normed=True)
    
    # Plot histogram on the current subplot
    ax.plot(hist)
    
    # Image Label setup
    ax.set_ylabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')

plt.subplots_adjust(hspace=0.3)

# Display the plot
plt.show()

<Figure size 6000x7500 with 5 Axes>

## Mask: HSV Model (Manually Adjust Hue and Tolerance Value)

In [91]:
def Control_Tolerance_Value(image, tol_lst, h = 0):
    assert len(tol_lst) == 5
    hsv = cv2.cvtColor(image.rgb, cv2.COLOR_BGR2HSV)
    hue = hsv[:,:,0]
    fig, axes = plt.subplots(1, 5, figsize=(12, 5)) 
    fig.subplots_adjust(hspace=0.5)
    
    for i, t in enumerate(tol_lst):
        ax = axes[i]
        min_hue = np.array([h - t])
        max_hue = np.array([h + t])
        
        mask_hue = cv2.inRange(hue, min_hue, max_hue)
        ax.imshow(mask_hue, cmap='gray')
        ax.axis('off')
        ax.set_title(f'Tolerance Value = {t}', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

### Hue = 0

In [92]:
for i in images:
    Control_Tolerance_Value(locals()[i], [10, 20, 30, 40, 60])

<Figure size 6000x2500 with 5 Axes>

<Figure size 6000x2500 with 5 Axes>

<Figure size 6000x2500 with 5 Axes>

<Figure size 6000x2500 with 5 Axes>

<Figure size 6000x2500 with 5 Axes>

### Hue = 20

In [93]:
for i in images:
    Control_Tolerance_Value(locals()[i], [10, 20, 30, 40, 60],20)

<Figure size 6000x2500 with 5 Axes>

<Figure size 6000x2500 with 5 Axes>

<Figure size 6000x2500 with 5 Axes>

<Figure size 6000x2500 with 5 Axes>

<Figure size 6000x2500 with 5 Axes>

## Method 1

In [94]:
eyeBrow_dict = {}
fig, axes = plt.subplots(1, 5, figsize=(12, 5))

for i, ax in enumerate(axes):
    img = locals()[images[i]].rgb
    
    # Selected area
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    x, y, w, h = 140, 310, 400, 120 
    mask[y:y+h, x:x+w] = 255 
    selected_area = cv2.bitwise_and(img, img, mask=mask)
    
    # Add to dict
    eyeBrow_dict[f"{images[i]}"] = selected_area
    
    ax.imshow(selected_area)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Image Label setup
    ax.set_xlabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')
        
plt.subplots_adjust(wspace=0.1)
plt.show()

<Figure size 6000x2500 with 5 Axes>

In [95]:
def Control_Tolerance_Value2(image, tol_lst, h = 0):
    assert len(tol_lst) == 5
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hue = hsv[:,:,0]
    fig, axes = plt.subplots(1, 5, figsize=(12, 5)) 
    fig.subplots_adjust(hspace=0.5)
    
    for i, t in enumerate(tol_lst):
        ax = axes[i]
        min_hue = np.array([h - t])
        max_hue = np.array([h + t])
        
        mask_hue = cv2.inRange(hue, min_hue, max_hue)
        ax.imshow(mask_hue, cmap='gray')
        ax.axis('off')
        ax.set_title(f'Tolerance Value = {t}', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

In [96]:
for key in eyeBrow_dict.keys():
    Control_Tolerance_Value2(eyeBrow_dict[key], [10, 20, 30, 40, 60],20)

<Figure size 6000x2500 with 5 Axes>

<Figure size 6000x2500 with 5 Axes>

<Figure size 6000x2500 with 5 Axes>

<Figure size 6000x2500 with 5 Axes>

<Figure size 6000x2500 with 5 Axes>

In [97]:
def Eyeshadow_Contour(image, selected_area, hue_range, tol, canny_val, kernel_val, min_contour_area):
    # Eyeshadow HSV
    hsv = cv2.cvtColor(selected_area, cv2.COLOR_BGR2HSV)
    hue = hsv[:,:,0]
    
    # Adjust hue and tolerance
    min_hue = np.array([hue_range - tol])
    max_hue = np.array([hue_range + tol])
    mask_hue = cv2.inRange(hue, min_hue, max_hue)
    color = (255, 255, 255) 
    colored_mask = np.zeros_like(selected_area)
    colored_mask[mask_hue == 255] = color
    result = cv2.bitwise_and(selected_area, colored_mask)
    
    # contour
    edged = cv2.Canny(result, canny_val[0], canny_val[1])
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, kernel_val)
    dilate = cv2.dilate(edged, kernel, iterations=1)

    # find the contours in the dilated image
    contours, hierarchy = cv2.findContours(dilate, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    image_copy = image.copy()
    
    # Define the minimum contour area threshold
    min_contour_area = min_contour_area

    # Iterate over the contours
    filtered_contours = []
    for contour in contours:
        # Calculate the area of the contour
        contour_area = cv2.contourArea(contour)

        # Check if the contour area is above the threshold
        if contour_area > min_contour_area:
            filtered_contours.append(contour)

    # Draw the filtered contours on a copy of the original image
    cv2.drawContours(image_copy, filtered_contours, -1, color=(0, 255, 0), thickness=cv2.FILLED)
    
    return image_copy


In [98]:
eyeshadow_contour = {}
eyeshadow_contour[images[0]] = Eyeshadow_Contour(locals()[images[0]].rgb, eyeBrow_dict[images[0]], 10, 30, (100,200), (2,2), 0)
eyeshadow_contour[images[1]] = Eyeshadow_Contour(locals()[images[1]].rgb, eyeBrow_dict[images[1]], 20, 10, (200, 200), (5, 5), 500)
eyeshadow_contour[images[2]] = Eyeshadow_Contour(locals()[images[2]].rgb, eyeBrow_dict[images[2]], 20, 10, (400, 400), (5, 5), 500)
eyeshadow_contour[images[3]] = Eyeshadow_Contour(locals()[images[3]].rgb, eyeBrow_dict[images[3]], 20, 20, (300,300), (2,2), 150)
eyeshadow_contour[images[4]] = Eyeshadow_Contour(locals()[images[4]].rgb, eyeBrow_dict[images[4]], 20, 20, (300, 400), (9, 4), 150)

In [99]:
fig, axes = plt.subplots(1, 5, figsize=(12, 5))
for i, ax in enumerate(axes):
    ax.imshow(eyeshadow_contour[images[i]])
    ax.set_xticks([])
    ax.set_yticks([])

    # Image Label setup
    ax.set_xlabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')
    
    # Adjust axis linewidth
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        
plt.subplots_adjust(wspace=0.1)
plt.show()

<Figure size 6000x2500 with 5 Axes>

## Method 2

In [100]:
def alphaBlend(alpha, foreground, background):
    fore = np.zeros(foreground.shape, dtype=foreground.dtype)
    fore = cv2.multiply(alpha, foreground, fore, 1 / 255.0)
    alphaPrime = np.ones(alpha.shape, dtype=alpha.dtype) * 255 - alpha
    back = np.zeros(background.shape, dtype=background.dtype)
    back = cv2.multiply(alphaPrime, background, back, 1 / 255.0)
    outImage = cv2.add(fore, back)
    return outImage

In [101]:
def EyeShadow(eyeshadow, original_img, hue_range=10, tol=30, dilate=(5,5)):
    hsv = cv2.cvtColor(eyeshadow, cv2.COLOR_BGR2HSV)
    hue = hsv[:,:,0]
    min_hue = np.array([hue_range - tol])
    max_hue = np.array([hue_range + tol])
    mask_hue = cv2.inRange(hue, min_hue, max_hue)
    color = (255, 255, 255) 
    colored_mask = np.zeros_like(original_img)
    maskSmall = cv2.dilate(colored_mask, (5, 5))
    maskSmall = cv2.GaussianBlur(maskSmall, (5, 5), 0, 0)
    maskSmall[mask_hue == 255] = color
    mask_region = np.zeros_like(img)
    x, y, w, h = 140, 310, 400, 120
    mask_region[y:y + h, x:x + w] = 255
    maskSmall = cv2.bitwise_and(maskSmall, mask_region)
    return maskSmall

In [102]:
def ChangeEyeShadow(img, color=0, hue_range=10, tol=30, dilate=(5,5)):
    # Assertion
    assert type(color) == tuple and len(color) == 3
    assert type(dilate) == tuple and len(dilate) == 2
    
    # Step 1: ROI - Eye shadow mask
    mask1 = np.zeros(img.shape[:2], dtype=np.uint8)
    x, y, w, h = 140, 310, 400, 120
    mask1[y:y+h, x:x+w] = 255 
    selected_area = cv2.bitwise_and(img, img, mask=mask)
    mask2 = EyeShadow(selected_area, img, hue_range, tol, (5,5))
    
    # Step 2: foreground - with desire color
    foreground = mask2.copy()
    foreground[np.where((foreground == (255, 255, 255)).all(axis=2))] = color
    foreground = cv2.bitwise_and(img, foreground)
    
    # Result
    result = alphaBlend(mask2, foreground, img)
    return result

In [103]:
eyeshadow_gree = {}
eyeshadow_gree[images[0]] = ChangeEyeShadow(locals()[images[0]].rgb, color = (0, 255, 0), hue_range=10, tol=30)
eyeshadow_gree[images[1]] = ChangeEyeShadow(locals()[images[1]].rgb, color = (0, 255, 0), hue_range=10, tol=30)
eyeshadow_gree[images[2]] = ChangeEyeShadow(locals()[images[2]].rgb, color = (0, 255, 0), hue_range=10, tol=30)
eyeshadow_gree[images[3]] = ChangeEyeShadow(locals()[images[3]].rgb, color = (0, 255, 0), hue_range=10, tol=30)
eyeshadow_gree[images[4]] = ChangeEyeShadow(locals()[images[4]].rgb, color = (0, 255, 0), hue_range=10, tol=30)

In [104]:
fig, axes = plt.subplots(1, 5, figsize=(12, 5))
for i, ax in enumerate(axes):
    ax.imshow(eyeshadow_gree[images[i]])
    ax.set_xticks([])
    ax.set_yticks([])

    # Image Label setup
    ax.set_xlabel(" ".join(word.capitalize() for word in images[i].split("_")), fontweight='bold')
    
    # Adjust axis linewidth
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        
plt.subplots_adjust(wspace=0.1)
plt.show()

<Figure size 6000x2500 with 5 Axes>